### Q1. What is the average, median, and standard deviation of house prices?

-->

Answer:
The average house price is ₹5,38,932, the median is ₹4,50,000, and the standard deviation is ₹3,67,532. The mean is higher than the median by ₹88,932, which means a few very expensive luxury houses are pulling the average upward. The high standard deviation (nearly 68% of the mean) confirms that prices are widely spread across the dataset — the "average" price is not a reliable reference point here. The median of ₹4,50,000 is a more realistic representation of a typical house price.

```python
import pandas as pd

df = pd.read_csv('House_Price_India.csv')

print("Average Price :", df['Price'].mean())
print("Median Price  :", df['Price'].median())
print("Std Deviation :", df['Price'].std())
```

---
---

### Q2. Which number of bedrooms is most common?

-->

Answer:
The most common number of bedrooms is 3, with 6,612 houses (45.23% of the dataset), followed by 4-bedroom homes with 4,724 houses (32.31%). Together, 3 and 4 bedroom homes make up over 77% of the entire market. This shows the dataset is dominated by standard family-sized homes. Very small (1 bedroom) and very large (7+ bedroom) homes are rare. This is typical of a residential real estate market catering primarily to nuclear families.

```python
bedroom_count = df['number of bedrooms'].value_counts()
print("Bedroom Distribution:\n", bedroom_count)
print("\nMost Common:", bedroom_count.idxmax(), "bedrooms")
```

---
---

### Q3. Check if the price data is skewed.

-->

Answer:
The Price column has a skewness of 4.27, which is strongly positive (right-skewed). A skewness above 1 is already considered highly skewed — 4.27 is far beyond that. This means most houses are priced in the lower range, but a small number of extremely expensive properties (up to ₹77,00,000) create a very long right tail. As a result, the mean is pulled well above the median. For machine learning purposes, a log transformation on the Price column would be necessary before using it as a target variable.

```python
print("Price Skewness :", df['Price'].skew())
print("Mean           :", df['Price'].mean())
print("Median         :", df['Price'].median())
print("Difference     :", df['Price'].mean() - df['Price'].median())
```

---
---

### Q4. What is the average price for each number of bedrooms?

-->

Answer:
Average price rises consistently from 1 to 8 bedrooms — from ₹3,08,964 for 1-bedroom homes up to ₹12,08,455 for 8-bedroom homes. This makes sense as more bedrooms mean more space and higher demand from larger families or premium buyers. However, after 8 bedrooms the trend breaks — 9 and 11 bedroom averages drop. This is because only 3–4 records exist in those categories, making their averages statistically unreliable. The one record with 33 bedrooms is almost certainly a data entry error.

```python
avg_by_bed = df.groupby('number of bedrooms')['Price'].mean()
count_by_bed = df['number of bedrooms'].value_counts().sort_index()

print("Average Price by Bedrooms:\n", avg_by_bed)
print("\nRecord Count per Category:\n", count_by_bed)
```

---
---

### Q5. What is the relationship between living area and price? (Area without Basement)

-->

Answer:
The correlation between Area of the house (excluding basement) and Price is 0.615 — a moderate to strong positive relationship. This means larger homes generally cost more, but area alone does not fully determine price. For comparison, total living area (including basement) has a slightly stronger correlation of 0.712, and grade of the house also correlates at 0.671. This shows that area is an important factor but must be combined with grade, location, and waterfront status for accurate price estimation.

```python
print("Area (excl. basement) vs Price :", df['Area of the house(excluding basement)'].corr(df['Price']))
print("Total Living Area vs Price     :", df['living area'].corr(df['Price']))
print("Grade vs Price                 :", df['grade of the house'].corr(df['Price']))
```

---
---

### Q6. Identify any anomalies where houses have high prices but low areas.

-->

Answer:
Using the 75th percentile of Price (₹6,45,000) as the high-price threshold and the 25th percentile of Area excluding basement (1,200 sq ft) as the low-area threshold, 139 anomalous records were found. These are small houses priced above ₹6,45,000. Analysing these records shows their average grade is 7.29, about 5% have waterfront access, and most are near 2 schools. This confirms these high prices are driven by location, waterfront, and quality — not size. In real estate, a prime location can easily override the impact of small square footage.

```python
price_75 = df['Price'].quantile(0.75)
area_25 = df['Area of the house(excluding basement)'].quantile(0.25)

anomalies = df[(df['Price'] > price_75) & (df['Area of the house(excluding basement)'] < area_25)]

print("Anomalies Found:", len(anomalies))
print(anomalies[['Price', 'Area of the house(excluding basement)',
                  'grade of the house', 'waterfront present']].head(10))
```

---
---

### Q7. Compare average price based on number of floors and houses with or without waterfront.

-->

Answer:
By floors: prices rise from ₹4,36,977 (1 floor) to ₹6,48,737 (2 floors), with 2.5 and 3.5 floor homes peaking above ₹11,00,000 — though these have fewer records. More floors generally means a larger, premium property. By waterfront: non-waterfront homes average ₹5,30,417 while waterfront homes average ₹16,41,902 — a 3x premium. Only 112 out of 14,620 houses have waterfront access (0.8%), yet they command the highest prices of any category. Waterfront presence is the most powerful binary price driver in this dataset.

```python
print("Average Price by Floors:")
print(df.groupby('number of floors')['Price'].mean())

print("\nAverage Price by Waterfront (0=No, 1=Yes):")
print(df.groupby('waterfront present')['Price'].mean())

print("\nWaterfront Count:")
print(df['waterfront present'].value_counts())
```

---
---

### Q8. Identify the minimum and maximum house price. What does this indicate?

-->

Answer:
The minimum price is ₹78,000 — a 2-bedroom, 780 sq ft, single-floor home built in 1942 with a grade of 5 (poor quality) and condition of 1 (very poor), never renovated. The maximum price is ₹77,00,000 — a 6-bedroom, 12,050 sq ft luxury home with 8 bathrooms, grade 13, renovated in 1987. The ratio between the two is nearly 1:99, meaning the most expensive house costs almost 100 times more than the cheapest. This extreme range confirms this is a highly segmented market where budget, mid-range, and ultra-luxury properties co-exist and cannot be analysed with a single average.

```python
print("Minimum Price:", df['Price'].min())
print("Maximum Price:", df['Price'].max())

print("\nCheapest House:")
print(df.loc[df['Price'].idxmin(), ['Price','number of bedrooms','living area','grade of the house','condition of the house']])

print("\nMost Expensive House:")
print(df.loc[df['Price'].idxmax(), ['Price','number of bedrooms','living area','grade of the house','Renovation Year']])
```

---
---

### Q9. Which location (zipcode/area) has the highest average price?

-->

Answer:
Postal Code 122071 has the highest average price at ₹23,48,311 — 4.36 times the overall dataset average of ₹5,38,932. The next four are 122048 (₹12,96,414), 122057 (₹11,88,517), 122047 (₹10,69,295), and 122061 (₹8,89,590). All top postal codes average well above ₹8,00,000, while several lower-ranked areas average below ₹5,00,000. This confirms that location is one of the strongest price drivers — the same house in postal code 122071 could be worth 4 to 5 times more than in a lower-ranked area.

```python
location_avg = df.groupby('Postal Code')['Price'].agg(['mean','count']).sort_values('mean', ascending=False)

print("Top 10 Postal Codes by Average Price:")
print(location_avg.head(10))
print("\nOverall Average:", df['Price'].mean())
```

---
---

### Q10. Write at least 5 insights from your analysis.

-->

Answer:
Based on the complete EDA of 14,620 house records, here are 6 key insights:

**Insight 1 — Price is strongly right-skewed (skewness = 4.27):** Most houses are affordable but a few luxury properties push the mean far above the median. The median (₹4,50,000) is a more honest price reference than the mean (₹5,38,932).

**Insight 2 — 3 and 4 bedroom homes dominate the market (77% of records):** The market is clearly built around nuclear family housing. Investors targeting the highest-demand segment should focus on 3–4 bedroom properties.

**Insight 3 — Waterfront homes cost 3x more than non-waterfront:** Despite being only 0.8% of the dataset, waterfront homes average ₹16,41,902 vs ₹5,30,417. It is the most powerful single binary feature in the dataset.

**Insight 4 — Location creates a 4x price gap:** Postal Code 122071 averages ₹23,48,311 while lower-ranked areas fall below ₹5,00,000. Location must always be the first variable considered when evaluating any property.

**Insight 5 — Area alone cannot explain price (139 small homes are priced high):** 139 records with small area (below 1,200 sq ft) are still priced above ₹6,45,000, driven by waterfront, schools, and grade. Price is always a multi-factor outcome.

**Insight 6 — Grade of house is a key quality signal (correlation = 0.67):** Grade ranges from 5 (cheapest house) to 13 (most expensive). It is one of the strongest individual correlates of price and should always be included in any pricing model.

```python
print("Skewness                  :", df['Price'].skew())
print("Most common bedrooms      :", df['number of bedrooms'].value_counts().idxmax())
print("Waterfront premium        :", round(df[df['waterfront present']==1]['Price'].mean() /
                                           df[df['waterfront present']==0]['Price'].mean(), 2), "x")
print("Top ZIP avg               :", df.groupby('Postal Code')['Price'].mean().max())
print("Grade vs Price corr       :", round(df['grade of the house'].corr(df['Price']), 3))
```

---
---